# Regulatory capital: FRTB SA, SA-CCR, and Initial Margin

**Purpose:** Demonstrate standardized market-risk capital, counterparty-credit exposure, and initial-margin calculations exposed by `finstack_quant.margin`.

**Prerequisites:** `07_advanced_quant/margin_collateral_and_xva.ipynb` and comfort with rates/credit/equity/FX sensitivities.

**What you'll learn:**

- Build FRTB SBA sensitivity sets and compare engine versus function entry points.
- Compute SA-CCR exposure-at-default for a derivatives netting set.
- Inspect SIMM, schedule, and haircut initial-margin results.

This notebook demonstrates standardized regulatory capital and initial-margin calculations exposed by `finstack_quant.margin`:

- **FRTB SBA** (Fundamental Review of the Trading Book, Sensitivities-Based Approach) -- market-risk capital from delta / vega / curvature sensitivities across risk classes, with default risk charge (DRC) and the residual-risk add-on (RRAO).
- **SA-CCR** (Standardised Approach for Counterparty Credit Risk) -- the exposure-at-default (EAD) of a derivatives netting set as `alpha * (RC + PFE)`.
- **Initial margin** -- computable SIMM, schedule, and haircut examples returning real `ImResult` objects.

All inputs and results are plain numbers / dictionaries or typed Python wrappers around Rust results, so notebook code stays inspectable while calculations run in the core library.

## Imports

In [ ]:
import json

import pandas as pd

from finstack_quant.margin import (
    CollateralAssetClass,
    FrtbSensitivities,
    FrtbSbaEngine,
    HaircutImCalculator,
    NettingSetId,
    SaCcrEngine,
    SaCcrNettingSetConfig,
    SaCcrTrade,
    ScheduleImCalculator,
    SimmCalculator,
    SimmSensitivities,
    frtb_sba_charge,
    saccr_ead,
)

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

## FRTB SBA: building a sensitivity set

`FrtbSensitivities` collects standardized sensitivities by risk class. Conventions:

- **GIRR delta** is reported as base-currency P&L per a **1 percentage-point** parallel rate move at each canonical tenor (roughly `100 x DV01`). Tenor labels are case-sensitive (`"2Y"`, `"5Y"`, `"10Y"`, ...).
- **Vega** is supplied per option maturity / underlying tenor; **curvature** as the up / down CVR shocks.
- **RRAO** notional carries a 1% weight for exotic instruments, 0.1% otherwise.

Below is a rates-led trading book with some credit, equity, and FX delta.

In [ ]:
sens = FrtbSensitivities("USD")

# GIRR (interest-rate) delta: base-ccy P&L per 1pp parallel shift at each tenor
sens.add_girr_delta("2Y", 12_000.0)
sens.add_girr_delta("5Y", 25_000.0)
sens.add_girr_delta("10Y", -8_000.0)

# Credit-spread (non-securitisation) delta
sens.add_csr_nonsec_delta("ACME_CORP", 1, "5Y", 4_000.0)

# Equity delta (single name, bucket 1) and FX delta vs reporting currency
sens.add_equity_delta("ACME_CORP", 1, 10_000.0)
sens.add_fx_delta("EUR", "USD", 15_000.0)

# GIRR vega and curvature
sens.add_girr_vega("1Y", "5Y", 5_000.0)
sens.add_girr_curvature(2_000.0, -1_800.0)

# Residual-risk add-on for an exotic
sens.add_rrao_position("EXOTIC_TARN", 1_000_000.0, is_exotic=True)

sens.base_currency

## FRTB SBA: the capital charge

`frtb_sba_charge` returns a typed `FrtbSbaResult`. With no fixed correlation scenario it evaluates the low / medium / high correlation scenarios and reports the **binding** (maximum) one, then adds DRC and RRAO on top.

In [ ]:
result = frtb_sba_charge(sens)
total = result.total

print(f"Total SBA charge: {total:,.0f} {sens.base_currency}")
print(f"Binding correlation scenario: {result.binding_scenario}")
print(f"DRC: {result.drc:,.0f}   RRAO: {result.rrao:,.0f}")

In [ ]:
rows = []
for component in ("delta", "vega", "curvature"):
    for risk_class, charge in getattr(result, f"{component}_by_risk_class").items():
        rows.append({"component": component, "risk_class": risk_class, "charge": charge})

charge_df = pd.DataFrame(rows).sort_values(["component", "risk_class"]).reset_index(drop=True)
charge_df

### Correlation scenarios

The SBA recomputes aggregation under three prescribed correlation levels. The binding scenario drives the capital number; DRC and RRAO are then added on top.

In [ ]:
scenario_charges = result.scenario_charges
binding = result.binding_scenario

scen_df = pd.DataFrame(
    [{"scenario": s, "sba_charge": scenario_charges[s], "binding": s == binding} for s in ("low", "medium", "high")]
)

reconstructed = scenario_charges[binding] + result.drc + result.rrao
print(f"binding SBA + DRC + RRAO = {reconstructed:,.0f}  (matches total: {abs(reconstructed - total) < 1e-6})")
scen_df

In [ ]:
# Object-oriented entry point; pin a single correlation scenario explicitly
high_total = FrtbSbaEngine(scenarios=["high"]).calculate(sens).total
print(f"FrtbSbaEngine(scenarios=['high']) total: {high_total:,.0f}")

## SA-CCR: trades and netting set

`SaCcrTrade(...)` (keyword arguments) and `SaCcrTrade.from_json` both validate the complete canonical derivative payload (asset class, notional, start / end date, underlier, hedging set, direction, supervisory delta, option classification, and current mark); `SaCcrTrade.from_dataframe` loads a whole trade tape. A netting set is keyed by a `NettingSetId` (bilateral or cleared) and configured as either **unmargined** or **margined** (with threshold, MTA, NICA, and an MPOR in business days).

In [ ]:
ir_swap = SaCcrTrade(
    trade_id="IRS_5Y", asset_class="interest_rate",
    notional=100_000_000.0, start_date="2024-01-15", end_date="2029-01-15",
    underlier="USD", hedging_set="USD_IRS",
    direction=1.0, supervisory_delta=1.0, mtm=2_500_000.0,
)
fx_fwd = SaCcrTrade.from_json(json.dumps({
    "trade_id": "FXFWD_EURUSD", "asset_class": "foreign_exchange",
    "notional": 50_000_000.0, "start_date": "2024-01-15",
    "end_date": "2025-07-15", "underlier": "EURUSD",
    "hedging_set": "EUR_USD", "direction": -1.0,
    "supervisory_delta": -1.0, "mtm": -500_000.0,
    "is_option": False, "option_type": None,
}))
trades = [ir_swap, fx_fwd]
[(t.trade_id, t.asset_class, f"{t.notional:,.0f}", f"{t.mtm:,.0f}") for t in trades]

## SA-CCR: EAD for an unmargined netting set

`SaCcrEngine.calculate_ead` returns replacement cost (RC), potential future exposure (PFE), the PFE multiplier, the aggregate add-on (and its breakdown by asset class), the supervisory `alpha`, and `EAD = alpha * (RC + PFE)`.

In [ ]:
unmargined = SaCcrNettingSetConfig.unmargined(
    NettingSetId.bilateral("MEGABANK", "CSA_UNCOLL"), collateral=0.0, as_of="2024-01-15"
)
res_u = SaCcrEngine().calculate_ead(unmargined, trades)

print(f"alpha = {res_u.alpha:.2f}   multiplier = {res_u.multiplier:.4f}")
print(f"RC = {res_u.rc:,.0f}   PFE = {res_u.pfe:,.0f}   EAD = {res_u.ead:,.0f}")
res_u.to_add_on_dataframe()

## SA-CCR: margining reduces EAD

A CSA with collateral, a threshold, MTA, and a margin period of risk (MPOR) lowers both RC (collateral offsets exposure) and PFE (only the MPOR window is at risk).

In [ ]:
margined = SaCcrNettingSetConfig.margined(
    NettingSetId.bilateral("MEGABANK", "CSA_COLL"),
    collateral=1_000_000.0,
    threshold=500_000.0,
    mta=250_000.0,
    nica=0.0,
    mpor_days=10,  # business days
    as_of="2024-01-15",
)
res_m = SaCcrEngine().calculate_ead(margined, trades)

pd.DataFrame([
    {"netting_set": "unmargined", "rc": res_u.rc, "pfe": res_u.pfe, "ead": res_u.ead},
    {"netting_set": "margined", "rc": res_m.rc, "pfe": res_m.pfe, "ead": res_m.ead},
])

In [ ]:
# Thin wrapper over SaCcrEngine.calculate_ead (alpha = 1.4 unless overridden)
flat = saccr_ead(trades, unmargined)
rc, pfe, ead = flat.rc, flat.pfe, flat.ead
print(f"saccr_ead -> RC={rc:,.0f}  PFE={pfe:,.0f}  EAD={ead:,.0f}")

## Initial margin: SIMM, schedule, and haircut

The Python bindings now expose direct calculator paths for the common IM methods. Each call returns an `ImResult` with the headline amount, currency, methodology, MPOR, calculation date, and risk-class / asset-class breakdown.

In [ ]:
def im_result_row(label, result):
    keys = result.breakdown_keys()
    return {
        "method": label,
        "amount": result.amount,
        "currency": result.currency,
        "methodology": str(result.methodology),
        "mpor_days": result.mpor_days,
        "as_of": result.as_of,
        "breakdown_keys": ", ".join(keys),
        "largest_breakdown": max((result.breakdown_amount(k) or 0.0) for k in keys) if keys else 0.0,
    }

simm_sens = SimmSensitivities("USD")
simm_sens.add_ir_delta("USD", "2Y", 12_000.0)
simm_sens.add_ir_delta("USD", "5Y", 25_000.0)
simm_sens.add_ir_vega("USD", "5Y", 5_000.0)
simm_sens.add_credit_qualifying_delta("financial", "BANK_A", "5Y", 8_000.0)
simm_sens.add_equity_delta("ACME_CORP", 10_000.0)
simm_sens.add_fx_delta("EUR", 15_000.0)

simm_im = SimmCalculator("v2_6").calculate_from_sensitivities(simm_sens, "USD", "2025-01-15")

schedule_im = ScheduleImCalculator.bcbs_standard().calculate_for_notional(
    100_000_000.0, "USD", "interest_rate", 5.0, "2025-01-15"
)

cash = CollateralAssetClass.cash()
haircut_im = HaircutImCalculator.bcbs_standard().calculate_for_collateral(
    10_000_000.0,
    "USD",
    cash,
    currency_mismatch=True,
    as_of="2025-01-15",
)

pd.DataFrame([
    im_result_row("SIMM", simm_im),
    im_result_row("Schedule", schedule_im),
    im_result_row("Haircut", haircut_im),
])

## Takeaways

- `frtb_sba_charge(sensitivities)` returns a `FrtbSbaResult`; its `.total` is the binding correlation scenario's SBA charge plus DRC and RRAO. `FrtbSbaEngine(scenarios=[...])` pins the scenarios (and `risk_classes=[...]` the risk classes).
- GIRR delta is base-currency P&L per 1pp move (~`100 x DV01`); pre-scale your sensitivities to the FRTB convention before loading them.
- `SaCcrEngine.calculate_ead(config, trades)` yields `RC`, `PFE`, the add-on breakdown, and `EAD = alpha * (RC + PFE)`; margined netting sets carry materially lower EAD than unmargined ones.
- `ImMethodology` selects the initial-margin regime (SIMM, schedule, haircut, internal model, clearing house).

### Methodology values

Every calculator above returns an `ImResult` whose regime is an `ImMethodology`. When the method arrives from configuration as a string, parse it with `ImMethodology.from_str` rather than comparing raw text.

In [ ]:
from finstack_quant.margin import ImMethodology

print("parsed from config string:", ImMethodology.from_str("simm"))
print("SIMM result reports:      ", simm_im.methodology)
print("Schedule result reports:  ", schedule_im.methodology)